In [1]:
print("HARI OM")

HARI OM


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import Bio

In [3]:
df=pd.read_csv("raw_data/HDMB.csv")

/var/folders/w_/hg7x5zzx15sg3m0gk991124m0000gn/T/ipykernel_4307/1379268035.py:1: DtypeWarning: Columns (3,4,7,8,9,10,11,12,13,14,15,16,18,24,27,30,31,32,33,37,38,47,48,49,50,51,52,54,62,64,66,67,68,69,74,79,80,81,82,83,85,86,89,90,91,92,93,94,95,96,97,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,130,131,132,135,136,141,142,143,147,149,152,153,157,158,159,160,161,162,163,168,170,171,172,173,177,179,180,181,182,183,184,185,186,187,188,189,190,191) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("raw_data/HDMB.csv")


In [4]:
df.head()

,library_id,project_id,sample_id,BioprojectID,PubmedID,project_name,HMgDB_sample_site_1,HMgDB_sample_site_2,HMgDB_sample_site_3,HMgDB_sample_site_4,...,assembled,sequence_count,basepairs_count,average_length,quality_above_30_SRA,Q30_SRA,drisee_score_raw_MGRAST,creation_date,update_date,source_database
0,SRR3340629,SRX1683607,SRS1378921,317435,NaN,ImMicroDyn,gut,stool,NaN,NaN,...,No,32283453.0,6.400000e+09,198.0,95.0,35.82,NaN,2017-05-02,2016-04-06,SRA
1,SRR3340631,SRX1683609,SRS1378922,317435,NaN,ImMicroDyn,gut,stool,NaN,NaN,...,No,28139242.0,5.560000e+09,198.0,93.0,35.64,NaN,2017-05-02,2016-04-06,SRA
2,ERR478998,ERX444830,ERS436676,266076,NaN,Potential of fecal microbiota for early stage ...,gut,stool,NaN,NaN,...,No,4222017.0,6.540000e+08,155.0,96.0,37.27,NaN,2014-11-02,2015-08-20,SRA
3,ERR478999,ERX444831,ERS436676,266076,NaN,Potential of fecal microbiota for early stage ...,gut,stool,NaN,NaN,...,No,4245124.0,6.630000e+08,156.0,96.0,37.33,NaN,2014-11-02,2015-08-20,SRA
4,ERR479006,ERX444838,ERS436724,266076,NaN,Potential of fecal microbiota for early stage ...,gut,stool,NaN,NaN,...,No,4824462.0,8.200000e+08,170.0,97.0,37.54,NaN,2014-11-02,2015-08-20,SRA


In [ ]:
from Bio import Entrez
Entrez.email = "dpant@scripps.edu"
handle = Entrez.efetch(db="sra", id="SRS1378922", rettype="runinfo", retmode="text")
# parse resulting XML/CSV for BioSample accession

HTTPError: HTTP Error 400: Bad Request

In [19]:
from Bio import Entrez
Entrez.email = "dpant@scripps.edu"

# Step 1: resolve accession to internal UID
search = Entrez.esearch(db="sra", term="ERS436676")
record = Entrez.read(search)
uid = record["IdList"][0]

# Step 2: fetch full record (XML, not runinfo — NCBI no longer reliably supports rettype=runinfo via efetch)
handle = Entrez.efetch(db="sra", id=uid, rettype="full", retmode="xml")
data = handle.read()
print(data)  # parse for <EXTERNAL_ID namespace="BioSample">...

b'<?xml version="1.0" encoding="UTF-8"  ?>\n<EXPERIMENT_PACKAGE_SET>\n<EXPERIMENT_PACKAGE><EXPERIMENT alias="ena-EXPERIMENT-EMBL - Germany-16-04-2014-11:28:14:514-42" center_name="EMBL - Germany" accession="ERX444831"><IDENTIFIERS><PRIMARY_ID>ERX444831</PRIMARY_ID></IDENTIFIERS><TITLE>Illumina HiSeq 2000 paired end sequencing</TITLE><STUDY_REF accession="ERP005534" refname="ena-STUDY-EMBL - Germany-08-04-2014-15:10:10:040-207" refcenter="EMBL - Germany"><IDENTIFIERS><PRIMARY_ID>ERP005534</PRIMARY_ID></IDENTIFIERS></STUDY_REF><DESIGN><DESIGN_DESCRIPTION/><SAMPLE_DESCRIPTOR refname="CCIS08668806ST-3-0" refcenter="EMBL - Germany" accession="ERS436676"><IDENTIFIERS><PRIMARY_ID>ERS436676</PRIMARY_ID><EXTERNAL_ID namespace="BioSample">SAMEA2466919</EXTERNAL_ID></IDENTIFIERS></SAMPLE_DESCRIPTOR><LIBRARY_DESCRIPTOR><LIBRARY_NAME>CCIS08668806ST-3-0_11s002725-1-1_lane8</LIBRARY_NAME><LIBRARY_STRATEGY>WGS</LIBRARY_STRATEGY><LIBRARY_SOURCE>METAGENOMIC</LIBRARY_SOURCE><LIBRARY_SELECTION>RANDOM</LIB

In [20]:
import requests

acc = "SAMEA112177524"
url = f"https://www.ebi.ac.uk/ena/portal/api/filereport?accession={acc}&result=read_run&fields=run_accession,sample_accession,secondary_sample_accession,study_accession&format=tsv"
resp = requests.get(url)
print(resp.text)

run_accession	sample_accession	secondary_sample_accession	study_accession
ERR10550333	SAMEA112177524	ERS14287364	PRJEB57593



In [18]:
import requests

def batch_query(accessions, batch_size=500):
    results = []
    for i in range(0, len(accessions), batch_size):
        batch = accessions[i:i+batch_size]
        query = " OR ".join(f'run_accession="{a}"' for a in batch)
        url = "https://www.ebi.ac.uk/ena/portal/api/search"
        params = {
            "result": "read_run",
            "query": query,
            "fields": "run_accession,sample_accession,secondary_sample_accession,study_accession",
            "format": "tsv"
        }
        resp = requests.get(url, params=params)
        results.append(resp.text)
    return results